# Hyperscanning EEG and BIDS: A Practical Proposal

**Authors:** Anne Monnier, Guillaume Dumas — Université de Montréal  
**Contribution to:** [BIDS Issue #402](https://github.com/bids-standard/bids-specification/issues/402)

---

## Context

**Hyperscanning** = simultaneous EEG recording of two or more participants during social interaction.  
This dataset: **mother–child dyads** (autistic × non-autistic), 10 tasks, dual EGI 128-channel EEG, synchronized via LSL/LabRecorder → XDF.

---

## The BIDS challenge

| Community proposal | Source | Limitation |
|--------------------|--------|------------|
| `ses-dyadic1` | Issue #402, BIDS FAQ | ❌ Conflicts with longitudinal sessions |
| `acq-dyad001` | Neurostars 2023 (Rémi Gau) | ⚠️ Semantic misuse of `acq` |
| **`dyad_id` in `participants.tsv`** | **This proposal** | ✅ Fully standard BIDS |

---

## Experimental protocol (10 tasks)

| Task | Duration |
|------|----------|
| 01 Rest eyes open | 1 min |
| 02 Rest eyes closed | 1 min |
| 03 Spontaneous imitation | 2 min |
| 04 Verbal (day planning) | 2 min |
| 05 Rest eyes open | 1 min |
| 06 Rest eyes closed | 1 min |
| 07 Verbal (day planning) | 2 min |
| 08 Spontaneous imitation | 2 min |
| 09 Rest eyes open | 1 min |
| 10 Rest eyes closed | 1 min |

## Part 1 — Source data structure

In [ ]:
from pathlib import Path

def print_tree(path, prefix='', max_depth=3, depth=0):
    if depth > max_depth:
        return
    items = sorted(Path(path).iterdir())
    for i, item in enumerate(items):
        connector = '└── ' if i == len(items)-1 else '├── '
        size = f'  ({item.stat().st_size} B)' if item.is_file() else ''
        print(prefix + connector + item.name + size)
        if item.is_dir():
            ext = '    ' if i == len(items)-1 else '│   '
            print_tree(item, prefix + ext, max_depth, depth+1)

print('sourcedata/')
print_tree('sourcedata')

## Part 2 — Reading the XDF file

The XDF contains all streams in a **shared LSL time reference** — this guarantees synchronization between the two EEG recordings. This is the source of temporal truth for the dyad.

In [ ]:
import pyxdf

streams, _ = pyxdf.load_xdf('sourcedata/dyad-001/EEG/dyad-001_eeg.xdf')

print(f'XDF streams ({len(streams)} found):\n')
for s in streams:
    info = s['info']
    print(f"  {info['name'][0]:25s}  type={info['type'][0]:10s}  ch={info['channel_count'][0]}")

print('\n→ All streams share the same LSL timestamp space.')
print('→ BIDS has no field to declare this link between sub-001 and sub-002.')
print('  This is an open question for the community.')

## Part 3 — IOS ratings

IOS = Inclusion of the Other in the Self (Aron et al. 1992), Likert 1–7, collected **after each task**.  
**Our proposal:** store as a custom column in `events.tsv` — not in `phenotype/` (which is for stable traits).

In [ ]:
import pandas as pd

ios_m = pd.read_csv('sourcedata/dyad-001/IOS/ios_sub-001.tsv', sep='\t')
ios_c = pd.read_csv('sourcedata/dyad-001/IOS/ios_sub-002.tsv', sep='\t')

merged = ios_m[['label','value']].rename(columns={'value':'mother_IOS'}).merge(
    ios_c[['label','value']].rename(columns={'value':'child_IOS'}), on='label'
)
print('IOS ratings per task — dyad-001:')
print(merged.to_string(index=False))

## Part 4 — BIDS rawdata structure

In [ ]:
print('rawdata/ (first subject only for clarity)')
print_tree('rawdata/sub-001', max_depth=3)
print('\n... same structure for sub-002 to sub-006')

## Part 5 — Our proposal: `dyad_id` in `participants.tsv`

In [ ]:
participants = pd.read_csv('rawdata/participants.tsv', sep='\t')
print('rawdata/participants.tsv')
print('=' * 60)
print(participants.to_string(index=False))
print()
print('dyad_id is a standard BIDS custom column.')
print('No .bidsignore needed. Validator compliant.')

## Part 6 — Querying dyads

In [ ]:
# Get participants of a dyad
def get_dyad(df, dyad_id):
    dyad = df[df['dyad_id'] == dyad_id]
    return dyad[dyad['role']=='mother'].iloc[0], dyad[dyad['role']=='child'].iloc[0]

mother, child = get_dyad(participants, 'dyad-002')
print(f'dyad-002:')
print(f'  Mother: {mother.participant_id} ({mother.group})')
print(f'  Child:  {child.participant_id} ({child.group})')
print()

# Get all autistic dyads
autistic = participants[participants['group']=='autistic']['dyad_id'].unique()
print(f'Autistic dyads: {list(autistic)}')
print()

# Scalable to any group size
print('All dyads:')
for dyad_id, grp in participants.groupby('dyad_id'):
    members = ' + '.join(f"{r.participant_id} ({r.role})" for _, r in grp.iterrows())
    print(f'  {dyad_id}: {members}')

## Part 7 — Events.tsv with IOS ratings

In [ ]:
# Show all 10 tasks for sub-001
all_events = []
for task_n in range(1, 11):
    task_label = [
        'restEyesOpen','restEyesClosed','imitation','verbal','restEyesOpen',
        'restEyesClosed','verbal','imitation','restEyesOpen','restEyesClosed'
    ][task_n-1]
    fname = f'rawdata/sub-001/ses-01/eeg/sub-001_ses-01_task-{task_n:02d}{task_label}_events.tsv'
    try:
        df = pd.read_csv(fname, sep='\t')
        all_events.append(df)
    except FileNotFoundError:
        pass

all_events_df = pd.concat(all_events, ignore_index=True)
print('events.tsv — sub-001, all 10 tasks:')
print(all_events_df[['onset','duration','trial_type','IOS_rating']].to_string(index=False))

## Part 8 — Why not the other proposals?

### `ses-dyadic1` — breaks longitudinal designs
```
# You want 3 timepoints:
sub-001/ses-01/  ← T1
sub-001/ses-02/  ← T2 (6 months)
sub-001/ses-03/  ← T3 (12 months)

# But with ses-dyadic1, ses is already used for the dyad → conflict!
```

### `acq-dyad001` — semantic misuse
`acq` is defined for **acquisition parameters** (resolution, sequence type).  
Dyad membership describes *who* was recorded together — not *how*.  
In practice: rare conflict, but semantically incorrect.

### `dyad_id` in `participants.tsv` ✅
Same pattern as `group`, `age`, `sex` — already standard.  
Works longitudinally. Scales to triads. Carries metadata.

## Part 9 — Open questions for the community

| Question | Our workaround | Status |
|----------|---------------|--------|
| Post-task ratings (IOS) | Custom column in `events.tsv` | ⚠️ Proposed |
| Shared temporal reference (XDF) | `scans.tsv` custom column | ⚠️ Open |
| Dyad-level derivatives (PLV) | `derivatives/` custom | ❌ No convention |
| Raw video (3 cameras, dyad-level) | `sourcedata/` only | ❌ No BEP |
| Post-hoc annotations (leader/follower) | `derivatives/video-annotation/` | ❌ No convention |

---

## References

- Poldrack et al. (2024). https://doi.org/10.1162/imag_a_00103  
- Pernet et al. (2019). https://doi.org/10.1038/s41597-019-0104-8  
- Appelhoff et al. (2019). https://doi.org/10.21105/joss.01896  
- Luke et al. (2025). https://www.nature.com/articles/s41597-024-04136-9  
- Monnier et al. (2025). https://doi.org/10.1093/nc/niaf052  
- BIDS Issue #402: https://github.com/bids-standard/bids-specification/issues/402